# 03 · VLM 視覺問答與看圖說話（VQA & Captioning）

> 模組：`05-Multimodal` ｜ 第 3 節 ｜ 生成端的起點
>
> 一句話定位：**VLM（Vision-Language Model）不是新物種，它就是你已經會的 LLM，前面接了一隻眼睛。** 圖片被編碼成一串「image token」插進序列，後面接上你早就熟悉的 `apply_chat_template`、`model.generate()`，其餘流程幾乎原封不動。

## 本 notebook 學習目標

讀完並跑完本節，你應該能夠：

1. 用一張圖說清楚 VLM 的三段式架構：**vision encoder → projector → LLM**，並理解 image token 如何被插入文字序列。
2. 用 `BitsAndBytesConfig` 4-bit 量化載入一個 7B VLM（呼應 `04-kbits-tuning` 的量化知識），並評估 VRAM 需求。
3. 用 `AutoProcessor` 處理**含影像的 messages**，理解 `[{'type':'image'}, {'type':'text', ...}]` 這個結構。
4. 把純文字 chatbot 的 `apply_chat_template(add_generation_prompt=True)` 直接延伸到多模態，完成 **VQA（視覺問答）** 與 **captioning（中文看圖說話）**。
5. 對照較輕量、無 chat 模板的 **BLIP-2** conditional generation 寫法，理解兩代 VLM 介面差異。
6. 玩進階的 **grounding / OCR**（Florence-2 與 Qwen2.5-VL 的 bbox 輸出）。
7. 管理**多輪含影像對話**的 history，並警覺 context window 與多模態幻覺問題。

## 前置知識（請先具備）

- **純文字生成與 chat 模板**：本節最關鍵的銜接點。若你還沒做過 `apply_chat_template` 與 `model.generate()`，先回去看 [`../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb`](../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb)。
- **量化載入**：4-bit / QLoRA 的原理，見 [`../../04-kbits-tuning/README.md`](../../04-kbits-tuning/README.md)。
- **`AutoProcessor` 抽象與共享嵌入空間**：本模組共用理論寫在模組 [`../README.md`](../README.md)；前一節 [`../02-clip_retrieval/clip_image_text_retrieval.ipynb`](../02-clip_retrieval/clip_image_text_retrieval.ipynb) 已示範 processor 同時吃影像與文字。

## 與既有模組的銜接

| 你在這裡用到的東西 | 你在哪一節已經學過 |
| :--- | :--- |
| `apply_chat_template` / `model.generate()` | [`../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb`](../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb) |
| `BitsAndBytesConfig` 4-bit 量化 | [`../../04-kbits-tuning/README.md`](../../04-kbits-tuning/README.md) |
| `AutoProcessor`（影像+文字一起前處理） | [`../02-clip_retrieval/clip_image_text_retrieval.ipynb`](../02-clip_retrieval/clip_image_text_retrieval.ipynb) |
| 共享嵌入 / 跨模態檢索（下一步整合） | [`../05-multimodal_rag/multimodal_embeddings_rag.ipynb`](../05-multimodal_rag/multimodal_embeddings_rag.ipynb) |
| 把本節的 VLM 拿去 LoRA 微調 | [`../06-vlm_finetuning/vlm_lora_finetune.ipynb`](../06-vlm_finetuning/vlm_lora_finetune.ipynb) |

本節從頭到尾只做一件事：**讓你看到『同一套 chat 模板機制』如何跨模態運作**。每跑一步，我都會提醒你「這一步在單模態版本你已經做過了」。

## 0. 版本鎖定（請第一個跑）

**WHY**：多模態模型對 `transformers` 版本特別敏感——新 VLM（Qwen2.5-VL、LLaVA-OneVision、Florence-2）的 modeling code 與 chat 模板常在小版本才補齊；processor 對 image token 的展開邏輯也跟著版本走。鎖定版本是避免「同樣的程式碼別人能跑你不能跑」的第一道防線。這份鎖定與全 repo 一致，**不另開一套**。

- `qwen-vl-utils` 是 Qwen-VL 系列的官方影像/視訊前處理工具（負責 smart resize、視訊抽幀），用 Qwen2.5-VL 時建議裝。
- 4-bit 量化必裝 `bitsandbytes`；CPU-only 環境裝不起來也沒關係，後面有輕量替代路線。

In [ ]:
# Pin versions consistent with the whole cookbook (do NOT run if your env is already set up).
# Uncomment to install in a fresh environment.

# !pip install -q \
#     "transformers>=4.46" \
#     "datasets>=3.0" \
#     "accelerate>=1.0" \
#     "bitsandbytes>=0.44" \
#     "qwen-vl-utils" \
#     "pillow" \
#     "requests"

# Quick environment sanity check
import transformers, torch, sys

print("python      :", sys.version.split()[0])
print("transformers:", transformers.__version__)  # expect >= 4.46
print("torch       :", torch.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"vram total  : {total_gb:.1f} GB")

## 1. VLM 架構心智模型：vision encoder + projector + LLM

在寫任何程式碼前，先建立心智模型——這是學多模態最划算的投資。

### 三段式管線

```
  ┌─────────────┐    ┌───────────┐    ┌──────────────────────────────┐
  │  影像 (PIL)  │ →  │  Vision   │ →  │  Projector / Connector        │
  │             │    │  Encoder  │    │  (MLP，把視覺特徵投影到        │
  │             │    │ (ViT/SigLIP)   │   LLM 的 token 嵌入維度)       │
  └─────────────┘    └───────────┘    └──────────────┬───────────────┘
                                                      │  N 個 image embedding
                                                      ▼
  「<image> 這張圖是什麼?」  →  tokenizer  →  [文字token...] + [N 個 image token] + [文字token...]
                                                      │
                                                      ▼
                                          ┌──────────────────────┐
                                          │   LLM (Qwen / Llama)  │ → 自回歸生成中文回答
                                          └──────────────────────┘
```

### 三個元件各做什麼

1. **Vision Encoder**（通常是 ViT 或 SigLIP）：把圖片切成 patch，輸出一組視覺特徵向量。**這就是你在 [`../01-image_classification`](../01-image_classification) 看過的 ViT**，差別只是這裡不接分類 head，而是把特徵往後送。
2. **Projector / Connector**（多半是一個小 MLP）：視覺特徵的維度通常跟 LLM 的詞嵌入維度不同，projector 負責「翻譯」——把視覺向量投影到 LLM 聽得懂的嵌入空間。**微調 VLM 時，這個 projector 常常是 `target_modules` 的重點**（見第 06 節）。
3. **LLM**（Qwen2.5 / Llama / OPT）：跟你在 02-Adv-tasks 用的文字模型是同一種東西，自回歸地把後續 token 生成出來。

### 關鍵洞察：image token 怎麼插進序列

這是整個 notebook 最重要的一句話：

> **圖片在 LLM 眼中，就是序列裡的一段「特殊 token」。** processor 會在你寫 `{'type':'image'}` 的位置，展開成 N 個 image placeholder token（例如 Qwen2.5-VL 的 `<|image_pad|>`、LLaVA 的 `<image>`）。模型 forward 時，再把這 N 個位置的嵌入「換成」projector 輸出的視覺嵌入。

所以——**LLM 的注意力機制根本不在乎這段嵌入是來自文字還是圖片**，它一視同仁。這就是為什麼「同一套 chat 模板、同一個 `generate()`」可以無痛跨模態：你只是在文字序列中間，多塞了一段來自眼睛的嵌入而已。

> 對照記憶：純文字時，`input_ids` 經過 embedding 層變成嵌入；多模態時，文字 token 一樣走 embedding 層，image token 的位置則被 vision 路徑的嵌入「覆蓋」。下游完全相同。

## 2. 用 4-bit 量化載入 7B VLM

**WHY 量化**：一個 7B VLM 以 bfloat16 載入約需 14-16 GB 純權重，加上 vision encoder、KV cache、影像特徵，很容易爆掉消費級 GPU。4-bit NF4 量化把權重壓到約 1/4，讓 7B VLM 能在 8-12 GB 上跑推論。**這個 `BitsAndBytesConfig` 你在 [`../../04-kbits-tuning`](../../04-kbits-tuning/README.md) 已經寫過——一字不改地搬過來**，這正是本模組的核心主張：量化邏輯不因模態改變。

### VRAM 警告（務必先讀）

| 模型 | 4-bit 推論概略 VRAM | 適用 |
| :--- | :--- | :--- |
| `Qwen/Qwen2.5-VL-7B-Instruct`（2026 首選，中文最強） | 約 7-9 GB | 12 GB+ GPU |
| `llava-hf/llava-onevision-qwen2-7b-ov-hf` | 約 7-9 GB | 12 GB+ GPU |
| `HuggingFaceTB/SmolVLM-Instruct`（輕量替代） | 約 2-4 GB | < 8 GB GPU |
| `Salesforce/blip2-opt-2.7b`（captioning 輕量對照） | 約 3-5 GB | < 8 GB GPU |

> 若你的 GPU < 12 GB，把下面的 `MODEL_ID` 換成 `HuggingFaceTB/SmolVLM-Instruct`，它同樣支援 chat 模板與 image token，介面幾乎一致，只是能力較弱。
>
> 若你**完全沒有 GPU**，本 notebook 的生成 cell 會極慢甚至 OOM；建議只讀程式碼理解流程，或改用 BLIP-2（第 6 節）這類更小的模型在 CPU 上跑 captioning。

In [ ]:
import torch
from transformers import BitsAndBytesConfig

# 2026 first choice: strong Chinese, native bbox/OCR. Swap to SmolVLM-Instruct if VRAM < 12GB.
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
# MODEL_ID = "HuggingFaceTB/SmolVLM-Instruct"            # lightweight fallback (<8GB)
# MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"  # alternative 7B VLM

# Identical 4-bit recipe you already used in 04-kbits-tuning. Nothing here is multimodal-specific.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                 # NF4 is the information-theoretic sweet spot for weights
    bnb_4bit_compute_dtype=torch.bfloat16,     # compute in bf16, store in 4-bit
    bnb_4bit_use_double_quant=True,            # quantize the quantization constants too (extra ~0.4GB saving)
)

print("Target model:", MODEL_ID)
print("Quantization:", bnb_config.quant_method, "| 4-bit NF4 + double quant")

**WHY 用 `AutoModelForImageTextToText`**：2026 的 `transformers` 把「圖+文進、文出」的生成式 VLM 統一到 `AutoModelForImageTextToText` 這個類別（取代過去零散的 `Qwen2VLForConditionalGeneration` 等專屬類別）。這跟你純文字時用 `AutoModelForCausalLM` 是平行的設計——**換個 Auto 類別，不換思維**。

載入參數也跟單模態一致：`device_map='auto'` 讓 accelerate 自動切分到可用裝置，`quantization_config` 餵入剛才那份 4-bit 設定。

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

# Model load: same from_pretrained signature you know, just a multimodal Auto class.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # 4-bit, identical to 04-kbits-tuning
    device_map="auto",                # let accelerate place layers automatically
    torch_dtype=torch.bfloat16,       # non-quantized parts (vision encoder, lm_head) stay bf16
)
model.eval()  # inference only in this notebook; no gradients needed

# AutoProcessor = tokenizer + image_processor bundled. This is the multimodal sibling of AutoTokenizer.
# min/max_pixels caps the number of image tokens (controls VRAM & speed for high-res images).
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,    # lower bound on visual tokens
    max_pixels=1280 * 28 * 28,   # upper bound; raise for OCR-heavy tasks, lower to save VRAM
)

print("Loaded:", type(model).__name__)
print("Processor:", type(processor).__name__)

## 3. 準備一張圖，看看 AutoProcessor 怎麼吃多模態 messages

**WHY 先看一張圖**：在丟進模型前，先確認「我們到底在問哪張圖」，避免後面 debug 時搞不清楚輸入。我們用一張公開測試圖（HuggingFace 的經典範例圖：兩隻貓躺在沙發上、旁邊有遙控器），方便你對照模型答得對不對。

In [ ]:
from PIL import Image
import requests

# Classic HF demo image: two cats on a couch with remotes. Easy to verify VQA correctness.
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(IMAGE_URL, stream=True).raw).convert("RGB")

print("image size (W, H):", image.size)
image  # display inline in Jupyter

### 多模態 messages 的結構

**WHY 這個結構是本節的靈魂**：純文字時，你的 `messages` 是這樣（你已經很熟）：

```python
messages = [{"role": "user", "content": "這張圖是什麼?"}]  # content 是純字串
```

多模態時，唯一的改變是：**`content` 從「字串」升級成「一個 list，裡面每個元素標明自己是 image 還是 text」**：

```python
messages = [{
    "role": "user",
    "content": [
        {"type": "image"},                                # 這裡會被展開成 N 個 image token
        {"type": "text", "text": "這張圖是什麼?"},        # 文字照常 tokenize
    ],
}]
```

就這樣。`role` 還是 `user`/`assistant`/`system`，整體還是一個 turn-by-turn 的 list。你會的 chat 模板心智模型，**100% 沿用**。

In [ ]:
# Multimodal messages: content is a LIST of typed parts. The only change from text-only chat.
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},  # placeholder; the actual PIL image is passed to processor() below
            {"type": "text", "text": "這張圖裡有什麼動物?牠們在做什麼?請用繁體中文回答。"},
        ],
    }
]

# Inspect the structure: same role-based list you know from text chatbots.
for turn in messages:
    print("role:", turn["role"])
    for part in turn["content"]:
        print("  part type:", part["type"], "->", part.get("text", "<image placeholder>"))

## 4. apply_chat_template → processor → model.generate()

**WHY 三步走**：這正是純文字 chatbot 的三步流程，只是中間的 tokenizer 換成 processor、多餵一個 `images` 參數。逐步對照：

| 步驟 | 純文字版（02-Adv-tasks 你做過） | 多模態版（本節） |
| :--- | :--- | :--- |
| 1. 套模板 | `tokenizer.apply_chat_template(msgs, add_generation_prompt=True)` | `processor.apply_chat_template(msgs, add_generation_prompt=True)` |
| 2. 前處理 | `tokenizer(text, return_tensors='pt')` | `processor(text=text, images=image, return_tensors='pt')` |
| 3. 生成 | `model.generate(**inputs, ...)` | `model.generate(**inputs, ...)` ← 完全相同 |

`add_generation_prompt=True` 的意義也一模一樣：在序列尾端補上「assistant 該開口了」的提示 token，讓模型知道現在輪到它生成。

**注意**：第 1 步我們先只產生「文字版的模板字串」（裡面已含 image placeholder token），第 2 步才把真正的 PIL 影像連同這個字串一起餵給 processor，由它負責把 placeholder 展開、對齊 `pixel_values`。**手動拼接幾乎必然踩到 token 數對不上的坑**（這也是 README 反覆強調用 processor 的原因）。

In [ ]:
# Step 1: apply chat template -> a prompt string containing the image placeholder token(s).
# Identical call to the text-only chatbot, just on the processor.
prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,                # return the raw templated string so we can inspect it
    add_generation_prompt=True,    # append the "assistant turn starts here" cue
)
print("=== Templated prompt (note the image placeholder token) ===")
print(prompt_text)

In [ ]:
# Step 2: processor turns (templated text + PIL image) into model-ready tensors.
# Output dict now carries BOTH input_ids/attention_mask (text) AND pixel_values + image_grid_thw (vision).
inputs = processor(
    text=[prompt_text],
    images=[image],
    return_tensors="pt",
    padding=True,
).to(model.device)

print("tensor keys:", list(inputs.keys()))
print("input_ids shape   :", tuple(inputs["input_ids"].shape))
print("pixel_values shape:", tuple(inputs["pixel_values"].shape))
# Count how many positions are the image placeholder -> these are the N image tokens from Section 1.
img_token_id = model.config.image_token_id if hasattr(model.config, "image_token_id") else None
if img_token_id is not None:
    n_img_tokens = (inputs["input_ids"][0] == img_token_id).sum().item()
    print(f"# image tokens in sequence: {n_img_tokens}  (these get replaced by vision embeddings)")

**WHY 用 `torch.inference_mode()` 與這組生成參數**：推論時關閉梯度可省記憶體、加速。生成參數的意義跟純文字完全相同（第 9 節會深入調參）：
- `max_new_tokens`：上限，避免無止盡生成。
- `do_sample=False`：先用 greedy/beam 之外最穩的 deterministic 解碼，讓你第一次跑就拿到可重現結果；要更有變化再開 sampling。

生成完拿到的是「prompt + 回答」的完整 id，要**切掉 prompt 部分**只留新生成的 token——這個 slice 技巧你在純文字 `generate` 也用過。

In [ ]:
# Step 3: model.generate() -- byte-for-byte the same API as text-only generation.
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,  # deterministic first; revisit sampling in Section 9
    )

# Trim the prompt tokens; keep only the newly generated answer (same slicing trick as text chatbots).
trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated_ids)]
answer = processor.batch_decode(
    trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print("=== VLM answer (zh-TW) ===")
print(answer)

## 5. VQA 與 Captioning：同一條管線，換個 prompt 就換任務

**WHY 把它包成函式**：上面三步每次都重複，封裝成一個 helper 之後，**VQA 與 captioning 的差別只剩 prompt 文字**——這正是要讓你體會的：在 VLM 世界裡，「任務」幾乎只是「prompt 工程」，底層管線不動。這跟純文字 instruction-tuned 模型「換 prompt 換任務」的直覺一致。

In [ ]:
def ask_vlm(image, question, max_new_tokens=256, **gen_kwargs):
    """One-shot VQA/captioning helper. Same 3-step pipeline, parameterized by the question text."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": question},
        ],
    }]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], return_tensors="pt", padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

In [ ]:
# --- VQA: ask a specific question about the image ---
print("[VQA] 數量問題:")
print(ask_vlm(image, "圖中有幾隻貓?牠們分別是什麼顏色?"))
print()
print("[VQA] 細節問題:")
print(ask_vlm(image, "沙發上除了貓以外,還有什麼物品?"))
print()
print("[VQA] 推理問題:")
print(ask_vlm(image, "從這張圖推測,這些貓現在是清醒還是在休息?為什麼?"))

In [ ]:
# --- Captioning: ask for a free-form description. Same pipeline, descriptive prompt. ---
print("[Caption] 簡短圖說:")
print(ask_vlm(image, "請用一句繁體中文描述這張圖片。", max_new_tokens=64))
print()
print("[Caption] 詳細圖說:")
print(ask_vlm(image, "請詳細描述這張圖片的內容、場景與氛圍,使用繁體中文。", max_new_tokens=256))

## 6. BLIP-2 對照：較輕量、無 chat 模板的 conditional generation

**WHY 看一個「上一代」VLM**：Qwen2.5-VL 是「chat-native」VLM，把影像塞進對話模板。但更早的 BLIP-2（2023）走的是**另一種介面**——沒有 chat 模板，直接 `processor(images, text)` 做 conditional generation。看懂這個對照，你才知道：

1. **不是所有 VLM 都吃 messages 結構**。讀別人的 repo 看到 `processor(images=..., text=...)` 直餵、沒有 `apply_chat_template`，多半就是 BLIP-2 這一輩。
2. BLIP-2 用 **Q-Former** 當 projector（取代簡單 MLP），把固定數量的 query 向量去「問」vision encoder——這是 projector 設計的另一種思路，呼應第 1 節。
3. BLIP-2 又小又快（2.7B），是 captioning 的輕量對照，沒有大 GPU 也能跑。

**對照表**：

| | Qwen2.5-VL（chat-native, 2026） | BLIP-2（conditional gen, 2023） |
| :--- | :--- | :--- |
| 介面 | `apply_chat_template` + messages | `processor(images, text)` 直餵 |
| projector | MLP connector | Q-Former（query-based） |
| 多輪對話 | 原生支援 | 不支援（單輪 conditional） |
| 中文 | 強 | 弱（OPT backbone 偏英文） |
| VRAM | 7-9 GB（4-bit） | 3-5 GB |

> 這正是本模組「Whisper vs Wav2Vec2」「SigLIP vs CLIP」式對照的精神：用世代差異照亮設計取捨。

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration

BLIP2_ID = "Salesforce/blip2-opt-2.7b"

# Note: this loads a SECOND model. If VRAM is tight, free Qwen first:
#   del model; import gc; gc.collect(); torch.cuda.empty_cache()
blip2_processor = Blip2Processor.from_pretrained(BLIP2_ID)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_ID,
    quantization_config=bnb_config,   # same 4-bit recipe; quantization is model-agnostic
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
blip2_model.eval()
print("BLIP-2 loaded:", type(blip2_model).__name__)

In [ ]:
# --- BLIP-2 unconditional captioning: NO chat template, just images in. ---
blip_inputs = blip2_processor(images=image, return_tensors="pt").to(blip2_model.device, torch.bfloat16)
with torch.inference_mode():
    cap_ids = blip2_model.generate(**blip_inputs, max_new_tokens=40)
print("[BLIP-2 caption]", blip2_processor.batch_decode(cap_ids, skip_special_tokens=True)[0].strip())

# --- BLIP-2 VQA: prompt is a plain string (prompt-engineered), still no messages structure. ---
blip_q = "Question: How many cats are in the picture? Answer:"
blip_inputs = blip2_processor(images=image, text=blip_q, return_tensors="pt").to(blip2_model.device, torch.bfloat16)
with torch.inference_mode():
    ans_ids = blip2_model.generate(**blip_inputs, max_new_tokens=20)
print("[BLIP-2 VQA]", blip2_processor.batch_decode(ans_ids, skip_special_tokens=True)[0].strip())
# Observe: BLIP-2 answers in English even when you want zh-TW -> why Qwen2.5-VL is the 2026 zh choice.

## 7. 進階：Grounding 與 OCR（bbox 輸出）

**WHY grounding/OCR 很重要**：純 captioning 只告訴你「圖裡有貓」，grounding 還能告訴你「貓在哪個座標」。這是 VLM 從「描述」走向「定位 / agent 操作 / 文件理解」的關鍵能力。有兩條路：

- **路線 A — Qwen2.5-VL 直接用 prompt 要 bbox**：Qwen2.5-VL 原生支援目標定位,只要在 prompt 裡要求它輸出座標(通常是 JSON 格式),就能拿到 `bbox_2d`。
- **路線 B — Florence-2 用 task token**:Florence-2 是專門的 grounding/OCR/detection 模型,介面跟前面都不同——用 `<OD>`、`<OCR>`、`<CAPTION>` 這類 **task prompt token** 觸發不同能力,輸出再用 `post_process_generation` 解析成結構化座標。

先看路線 A(直接複用我們已載入的 Qwen)。

In [ ]:
# Route A: ask Qwen2.5-VL for grounding via a structured-output prompt.
grounding_q = (
    "請偵測圖中所有的貓,以 JSON 陣列回傳,每個元素含 'bbox_2d' (左上x,左上y,右下x,右下y) "
    "與 'label' 欄位。只輸出 JSON,不要多餘文字。"
)
raw = ask_vlm(image, grounding_q, max_new_tokens=256)
print("[Qwen2.5-VL grounding raw output]")
print(raw)

# The output is JSON-ish text; parse it. Real code should guard against malformed JSON.
import json, re
match = re.search(r"\[.*\]", raw, re.DOTALL)
if match:
    try:
        boxes = json.loads(match.group(0))
        print("\nparsed boxes:", boxes)
    except json.JSONDecodeError:
        print("\n(could not parse as strict JSON -- VLMs need output validation, see Section 9)")

**WHY 路線 B 值得學**:Florence-2 雖然小(0.7B),但它把 detection、OCR、region caption 等任務做成「**task token 觸發**」的統一介面,輸出是結構化座標而非自由文字——對需要穩定 bbox/OCR 的生產場景,比靠大 VLM 「希望它輸出合法 JSON」可靠得多。它也示範了「同一個 processor 抽象,但 task 由特殊 token 驅動」這種第三種介面風格。

> Florence-2 需要 `trust_remote_code=True`(自帶 modeling code)。它較小,通常不需量化即可在 4-6 GB 上跑。

In [ ]:
from transformers import AutoModelForCausalLM as _AutoCausalLM  # Florence-2 registers itself this way

FLORENCE_ID = "microsoft/Florence-2-large"

# Florence-2 ships custom modeling code -> trust_remote_code=True. Small enough to run unquantized.
florence_model = _AutoCausalLM.from_pretrained(
    FLORENCE_ID, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto"
).eval()
florence_processor = AutoProcessor.from_pretrained(FLORENCE_ID, trust_remote_code=True)

def run_florence(image, task_token, text_input=""):
    """Florence-2 is driven by task tokens like <OD>, <OCR>, <CAPTION>, not chat templates."""
    prompt = task_token + text_input
    inp = florence_processor(text=prompt, images=image, return_tensors="pt").to(
        florence_model.device, torch.bfloat16
    )
    with torch.inference_mode():
        ids = florence_model.generate(
            input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
            max_new_tokens=1024, num_beams=3, do_sample=False,
        )
    text = florence_processor.batch_decode(ids, skip_special_tokens=False)[0]
    # post_process_generation parses raw text into structured {labels, bboxes}.
    return florence_processor.post_process_generation(
        text, task=task_token, image_size=(image.width, image.height)
    )

# Object detection: returns bboxes + labels as structured dict.
print("[Florence-2 <OD> object detection]")
print(run_florence(image, "<OD>"))

In [ ]:
# Florence-2 OCR demo: needs a text-bearing image. Reuse OD plumbing with the <OCR> task token.
# (Using the cats image; OCR will likely find nothing -> that's the expected, honest result.)
print("[Florence-2 <OCR>]")
print(run_florence(image, "<OCR>"))
print()
print("[Florence-2 <DETAILED_CAPTION>]")
print(run_florence(image, "<DETAILED_CAPTION>"))

# Takeaway: three interface styles seen so far --
#   Qwen2.5-VL : chat template + prompt-engineered structured output
#   BLIP-2     : direct conditional generation, no template
#   Florence-2 : task tokens (<OD>/<OCR>/...) + post_process_generation
# Same AutoProcessor abstraction underneath; pick the tool that fits the job.

## 8. 多輪含影像對話:history 管理與 context window

**WHY 多輪是另一個「你已經會」的延伸**:純文字 chatbot 的多輪,就是把 `{'role':'assistant','content':...}` 接回 `messages` list 再送一次。多模態完全相同——只是 user turn 的 content 變成含 `{'type':'image'}` 的 list。模型會記得前面那張圖,讓你接著追問。

但多模態有兩個**單模態沒有的陷阱**,務必警覺:

1. **Image token 吃掉大量 context**:一張高解析圖可能展開成數百到上千個 image token。多輪如果每輪都塞新圖,context window 會以驚人速度被填滿——這比純文字對話消耗快得多。
2. **多圖混淆**:當 history 裡有多張圖,模型可能把「這張圖」指代錯。寫 prompt 時最好明確標示「第一張圖 / 第二張圖」。

下面示範「同一張圖、連續追問」的 history 累積寫法。

In [ ]:
def chat_round(history, image_for_this_turn, user_text, max_new_tokens=256):
    """Append a user turn (optionally with an image), generate, append assistant reply. Returns updated history."""
    content = []
    if image_for_this_turn is not None:
        content.append({"type": "image"})
    content.append({"type": "text", "text": user_text})
    history = history + [{"role": "user", "content": content}]

    prompt = processor.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    # Collect ALL images referenced across the whole history, in order (processor needs them all).
    imgs = [img for img in _images_in_history if img is not None]
    inputs = processor(text=[prompt], images=imgs, return_tensors="pt", padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = processor.batch_decode(
        [o[len(i):] for i, o in zip(inputs["input_ids"], out)], skip_special_tokens=True
    )[0].strip()

    history = history + [{"role": "assistant", "content": [{"type": "text", "text": reply}]}]
    return history, reply

# Track images in turn order so processor can map each {'type':'image'} placeholder to a real PIL image.
_images_in_history = []
history = []

# Turn 1: show the image and ask.
_images_in_history.append(image)
history, r1 = chat_round(history, image, "這張圖裡有什麼?請用繁體中文簡短回答。")
print("[A1]", r1)

In [ ]:
# Turn 2: NO new image -> follow-up relies on the model remembering the earlier image (history carries it).
_images_in_history.append(None)  # this turn adds no image
history, r2 = chat_round(history, None, "那牠們躺在什麼上面?")
print("[A2]", r2)

# Turn 3: another follow-up, still grounded on the same image.
_images_in_history.append(None)
history, r3 = chat_round(history, None, "幫我把以上資訊整理成一句完整的繁體中文圖說。")
print("[A3]", r3)

# Context-window awareness: count tokens consumed so far. Image tokens dominate.
final_prompt = processor.apply_chat_template(history, tokenize=False, add_generation_prompt=False)
n_tok = len(processor.tokenizer(final_prompt)["input_ids"])
print(f"\nApprox text tokens in full history: {n_tok} (image tokens counted separately, often the larger share)")

## 9. 生成參數與多模態幻覺

**WHY 調參跟純文字一樣,但風險更高**:`temperature`、`top_p`、`repetition_penalty` 的意義跟你在文字生成學的完全相同——`temperature` 控隨機性、`top_p` 控核採樣截斷、`repetition_penalty` 抑制重複。但在 VLM 上,sampling 開太大會放大一個特有問題:**多模態幻覺(visual hallucination)**——模型「看圖說瞎話」,描述出圖裡根本沒有的物件。

幻覺的常見成因:
1. **語言先驗壓過視覺證據**:LLM backbone 太強,憑「沙發上通常有抱枕」就腦補抱枕,即使圖裡沒有。
2. **解析度不足**:image token 太少,細節丟失,模型用常識填補。
3. **sampling 溫度過高**:隨機性讓低機率(常是錯誤)的描述被選中。

**降低幻覺的實務手段**:
- VQA / 抽取任務用 `do_sample=False`(deterministic),不要為了「文采」開高溫。
- 加 `repetition_penalty`(約 1.05-1.1)避免複讀。
- prompt 明確要求「只根據圖中可見內容回答,不確定就說不確定」。
- 需要結構化輸出(如第 7 節 bbox)時,務必做輸出驗證(JSON parse + schema 檢查)。

In [ ]:
hallucination_probe = "請仔細描述這張圖。如果某個物件你不確定是否存在,請明說『不確定』,不要憑空臆測。"

print("=== A. Deterministic (recommended for VQA / extraction) ===")
print(ask_vlm(image, hallucination_probe, do_sample=False))

print("\n=== B. Low-temperature sampling (a bit more natural, still grounded) ===")
print(ask_vlm(
    image, hallucination_probe,
    do_sample=True, temperature=0.3, top_p=0.9, repetition_penalty=1.05,
))

print("\n=== C. High-temperature sampling (creative -> watch for hallucinated objects) ===")
print(ask_vlm(
    image, hallucination_probe,
    do_sample=True, temperature=1.2, top_p=0.95, repetition_penalty=1.1,
))
# Compare A/B/C: as temperature rises, descriptions drift further from what's actually in the image.

### 一個刻意的幻覺壓力測試

**WHY**:故意問一個「圖裡沒有的東西」,看模型會不會誠實說「沒有」還是順著問題編造。這是評估 VLM 可信度最快的探針——好的 2026 VLM 應該抗住這種誘導。

In [ ]:
# Leading question about a non-existent object. A trustworthy VLM should NOT confirm a dog.
print("[誘導性提問]")
print(ask_vlm(image, "圖中那隻狗是什麼品種?", do_sample=False))
# Desired behavior: the model points out there is no dog (only cats), instead of inventing a breed.

## 10. 小結與下一步

### 核心收穫

這一整份 notebook,其實只證明了一句話:

> **VLM 是 LLM 的多模態延伸,不是另起爐灶。** 你純文字學的 `apply_chat_template` + `model.generate()` + `BitsAndBytesConfig`,在這裡幾乎一字不改;唯一的新東西,是 `content` 從字串變成含 `{'type':'image'}` 的 list,以及背後「image token 插入序列」的機制。

逐項對照你這節做了什麼:

| 步驟 | 在單模態你已做過 | 多模態的唯一差異 |
| :--- | :--- | :--- |
| 4-bit 載入 | `04-kbits-tuning` 的 `BitsAndBytesConfig` | 換 `AutoModelForImageTextToText` |
| 前處理 | `AutoTokenizer` | `AutoProcessor`(多餵 `images=`) |
| 對話模板 | `apply_chat_template` | content 改成 typed list |
| 生成 | `model.generate()` | 完全相同 |
| 調參與風險 | temperature/top_p/repetition_penalty | 多了「視覺幻覺」這個維度 |

你也見識了**三種 VLM 介面風格**:Qwen2.5-VL(chat-native)、BLIP-2(conditional generation)、Florence-2(task token),理解了它們在 projector 設計與輸出形態上的取捨。

### 練習題

1. **換模型**:把 `MODEL_ID` 改成 `HuggingFaceTB/SmolVLM-Instruct`,重跑第 4-5 節。比較中文 captioning 品質與 VRAM 佔用,寫下差異。
2. **多圖比較**:再載入第二張圖(自選),用第 8 節的 history 機制,問模型「比較這兩張圖的差異」。觀察 context token 如何暴增,並驗證模型是否正確指代「第一張/第二張」。
3. **抗幻覺 prompt 工程**:針對第 9 節的誘導性提問,設計三種不同的 system/user prompt,量化哪種最能讓模型誠實回答「沒有狗」。
4. **結構化輸出穩定性**:把第 7 節路線 A(Qwen 要 JSON bbox)跑 10 次,統計有幾次輸出合法 JSON。思考:生產環境該如何保證解析穩定?(提示:輸出驗證 + 重試,或改用 Florence-2 路線 B。)
5. **OCR 實戰**:找一張含中文文字的圖(如菜單、招牌),分別用 Qwen2.5-VL(prompt 要 OCR)與 Florence-2(`<OCR>`)抽取文字,比較準確度。

### 通往下一份 notebook

你現在手上有了**生成端的 VLM**。接下來兩條路:

- **整合應用** → [`../05-multimodal_rag/multimodal_embeddings_rag.ipynb`](../05-multimodal_rag/multimodal_embeddings_rag.ipynb):把第 02 節學的跨模態檢索(找出相關圖片/文件)接上本節的 VLM 生成,組成多模態 RAG——「先檢索、再看圖回答」。
- **微調自己的 VLM** → [`../06-vlm_finetuning/vlm_lora_finetune.ipynb`](../06-vlm_finetuning/vlm_lora_finetune.ipynb):把本節載入的 4-bit Qwen2.5-VL,套上你在 [`../../03-PEFT/01-LoRA`](../../03-PEFT/01-LoRA/chatbot_lora.ipynb) 與 [`../../04-kbits-tuning/04-4bits_training`](../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb) 學的 LoRA/QLoRA,在自己的圖文資料上微調。屆時你會看到:`target_modules` 多含 projector,其餘訓練流程跟純文字 QLoRA 一模一樣——再次印證本模組的核心主張。

至此,生成弧線閉合:**載入 → 前處理 → chat 模板 → 生成 → 調參 → 多輪**,每一環都是你既有單模態知識的再應用。